In [ ]:
from torch.nn import *
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def unpack(a, n, m=2 ** torch.arange(16)):
    return (a[..., None] & m[:n]).bool().float()


def dat(k, c, len=0, bs=4, std=64):
    len = 2 ** int(np.log2((len or k + 1) * c))
    i = torch.randint(c, (bs, len))
    return torch.randn(bs, len, k).add_(torch.randn(bs, c, k).mul_(std)[torch.arange(bs)[:, None], i]), unpack(i, np.log2(c).__ceil__())


def Nn(i, o=0, m=8, b=1, bn=1, act=Hardswish(1)):
    o = o or i
    m *= max(i, o)
    return Sequential(Linear(i, m, not bn), *(BatchNorm1d(m, track_running_stats=0),) if bn else (), act, Linear(m, o, b)) if act else Linear(i, o, b)


class SelfAttentionNn(Module):
    def __init__(s, forget, In, out):
        super().__init__()
        s.forget = forget
        s.In = In
        s.out = out

    def forward(s, x):
        y = s.In(x * s.forget(x).sigmoid_())
        w = s.out(torch.cat((x, y), 1)).sigmoid_()
        return (x * w).add_(y.mul_(1 - w))


class SelfMultiAttentionNn(SelfAttentionNn):
    def forward(s, x):
        n = x.size(-1)
        x = torch.cat((x, s.In(x * s.forget(x).sigmoid_())), 1)
        return (x.view(-1, x.size(-1) // n, n) * s.out(x).view(-1, x.size(-1) // n, n).softmax(-2)).sum(-2)

In [ ]:
x, y = dat(16, 65536, bs=1)
x = x[0]
y = y[0]

In [ ]:
m = SelfAttentionNn(Nn(16, act=Tanh()), Nn(16, act=Tanh()), Nn(32, 16, act=Tanh()))
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.05, 2000)
s = torch.cuda.amp.GradScaler()
print(sum(i.numel() for i in m.parameters()), m)
for _ in range(20):
    for _ in range(100):
        o.zero_grad()
        with torch.cuda.amp.autocast():
            _ = m(x)
            l = functional.binary_cross_entropy_with_logits(_, y)
        s.scale(l).backward()
        s.step(o)
        s.update()
        O.step()
    print(l, (_.signbit() != y).all(1).sum() / len(y))

In [ ]:
%reset -f in out

In [ ]:
SelfAttentionNn(Linear(2,2),Linear(2,2),Linear(4,2))(torch.rand(8,2))